In [1]:
from pathlib import Path

import polars as pl

FILTERED_DIR = Path("../data/1_filtered")
TOKENIZED_DIR = Path("../data/2_tokenized")

## Stage 1: Filtered Games (UCI + evals + FEN)


In [2]:
# Load filtered games
filtered_files = sorted(FILTERED_DIR.glob("*.parquet"))[:3]
print("Filtered game files:", len(list(FILTERED_DIR.glob("*.parquet"))))
print("Sample files:", [f.name for f in filtered_files])

Filtered game files: 52
Sample files: ['filtered_2013-01.parquet', 'filtered_2013-02.parquet', 'filtered_2013-03.parquet']


In [3]:
# Load sample filtered game
filtered_sample = pl.read_parquet(filtered_files[0], n_rows=5)
print("Columns:", filtered_sample.columns)
print("\nSample row:")
filtered_sample

Columns: ['lichess_id', 'uci_moves', 'evals_cp', 'evals_raw', 'is_check', 'is_capture', 'piece_moved', 'promotion', 'is_en_passant', 'white_rating', 'black_rating', 'result', 'game_end_reason', 'time_initial', 'time_increment', 'utc_timestamp', 'opening', 'eco', 'ply_count', 'fen']

Sample row:


lichess_id,uci_moves,evals_cp,evals_raw,is_check,is_capture,piece_moved,promotion,is_en_passant,white_rating,black_rating,result,game_end_reason,time_initial,time_increment,utc_timestamp,opening,eco,ply_count,fen
str,str,list[i16],list[i16],list[bool],list[bool],list[str],list[str],list[bool],i16,i16,str,str,u16,u8,datetime[μs],str,str,u16,str
"""2irq4pg0""","""e2e4 e7e5 g1f3 d7d6 d2d4 e5d4 …","[12, 26, … null]","[12, 26, … 32767]","[false, false, … true]","[false, false, … true]","[""p"", ""p"", … ""q""]","["""", """", … """"]","[false, false, … false]",1785,1944,"""1-0""","""mate""",300,0,2013-01-01 06:15:59,"""Philidor Defense: Exchange Var…","""C41""",43,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"
"""cbgpp8cc""","""e2e4 e7e5 f1c4 b8c6 c2c3 g7g6 …","[22, 20, … null]","[22, 20, … -32768]","[false, false, … true]","[false, false, … false]","[""p"", ""p"", … ""p""]","["""", """", … """"]","[false, false, … false]",1538,1607,"""0-1""","""mate""",600,10,2013-01-01 20:08:24,"""Bishop's Opening""","""C23""",52,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"
"""8rpcvpav""","""e2e4 d7d5 e4d5 d8d5 b1c3 d5a5 …","[22, 41, … 0]","[22, 41, … 0]","[false, false, … true]","[false, false, … false]","[""p"", ""p"", … ""q""]","["""", """", … """"]","[false, false, … false]",1565,1598,"""1/2-1/2""","""agreement""",300,0,2013-01-02 02:45:21,"""Scandinavian Defense: Main Lin…","""B01""",89,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"
"""282s2n5n""","""e2e4 e7e5 c2c4 g8f6 d2d3 c7c6 …","[17, 30, … null]","[17, 30, … -32763]","[false, false, … false]","[false, false, … false]","[""p"", ""p"", … ""p""]","["""", """", … """"]","[false, false, … false]",1835,1812,"""0-1""","""resignation""",540,0,2013-01-02 02:58:57,"""English Opening: The Whale""","""C20""",106,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"
"""etvda2dw""","""e2e4 e7e5 g1f3 b8c6 f1c4 g8f6 …","[27, 37, … -1533]","[27, 37, … -1533]","[false, false, … false]","[false, false, … true]","[""p"", ""p"", … ""q""]","["""", """", … """"]","[false, false, … false]",1692,1866,"""0-1""","""resignation""",300,0,2013-01-02 11:48:16,"""Italian Game: Two Knights Defe…","""C57""",28,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"


## Stage 2: Tokenized (train/val ready)


In [4]:
# Load tokenized datasets
pretrain_path = TOKENIZED_DIR / "pretrain.parquet"
eval_path = TOKENIZED_DIR / "eval.parquet"

pretrain_sample = pl.read_parquet(pretrain_path, n_rows=1000)
eval_sample = pl.read_parquet(eval_path, n_rows=5)

print("Pretrain shape:", pl.read_parquet(pretrain_path).shape)
print("Eval shape:", pl.read_parquet(eval_path).shape)
print("\nPretrain columns:", pretrain_sample.columns)
print("\nSample tokenized row:")
pretrain_sample

Pretrain shape: (5148703, 1)
Eval shape: (5251, 1)

Pretrain columns: ['token_ids']

Sample tokenized row:


token_ids
list[u16]
"[0, 3, … 1]"
"[0, 4, … 1]"
"[0, 3, … 1]"
"[0, 4, … 1]"
"[0, 4, … 1]"
…
"[0, 3, … 1]"
"[0, 4, … 1]"
"[0, 3, … 1]"


## Token functions


In [5]:
from krasnal.tokens import (
    GAME_END_ID,
    GAME_START_ID,
    ID_TO_MOVE,
)

### Opening analysis


In [6]:
# Load filtered games and analyze openings
raw_lf = pl.scan_parquet(str(FILTERED_DIR / "*.parquet"))

openings = (
    raw_lf.select("opening")
    .collect()["opening"]
    .str.split(":")
    .list.get(0)
    .str.split(",")
    .list.get(0)
    .str.replace(r"#\d+", "")
    .str.strip_chars()
    .str.replace_all(r"\s+", " ")
    .unique()
    .sort()
)

print("unique normalized openings:", len(openings))
for opening in openings:
    print(opening)

unique normalized openings: 170
Alekhine Defense
Amar Opening
Amazon Attack
Amsterdam Attack
Anderssen Opening
Australian Defense
Barnes Defense
Barnes Opening
Benko Gambit
Benko Gambit Accepted
Benko Gambit Declined
Benoni Defense
Bird Opening
Bishop's Opening
Blackmar-Diemer
Blackmar-Diemer Gambit
Blackmar-Diemer Gambit Declined
Blumenfeld Countergambit
Blumenfeld Countergambit Accepted
Boden-Kieseritzky Gambit
Bogo-Indian Defense
Borg Defense
Borg Opening
Bronstein Gambit
Budapest Defense
Canard Opening
Caro-Kann Defense
Carr Defense
Catalan Opening
Center Game
Center Game Accepted
Clemenz Opening
Colle System
Crab Opening
Creepy Crawly Formation
Czech Defense
Danish Gambit
Danish Gambit Accepted
Danish Gambit Declined
Doery Defense
Duras Gambit
Dutch Defense
East Indian Defense
Elephant Gambit
English Defense
English Opening
English Orangutan
English Rat
Englund Gambit
Englund Gambit Complex
Englund Gambit Complex Declined
Englund Gambit Declined
Formation
Four Knights
Four Knights

### Sequence statistics


In [7]:
# length of the longest game in raw data (by move count)
raw_with_lengths = raw_lf.select(
    pl.col("uci_moves").str.split(" ").list.len().alias("move_count"),
    pl.col("uci_moves"),
).collect()

max_length = raw_with_lengths["move_count"].max()
max_idx = raw_with_lengths["move_count"].arg_max()
longest_game_moves = raw_with_lengths[max_idx, "uci_moves"]

print("longest game length (moves):", max_length)
print("example longest game:")
print(longest_game_moves)

longest game length (moves): 597
example longest game:
d2d4 d7d5 c2c4 c7c6 b1c3 c8f5 g1f3 g8f6 e2e3 h7h6 f1e2 e7e6 e1g1 b8d7 b2b3 f8e7 c1b2 f6e4 a1c1 d7f6 c4d5 c6d5 c3e4 f5e4 f3e5 e8g8 a2a3 f6d7 b3b4 d7e5 d4e5 b7b6 b2d4 a8c8 d1a4 c8c1 f1c1 d8b8 a4d7 f8e8 e2b5 a7a5 d7e8 b8e8 b5e8 a5b4 a3b4 e7b4 d4b6 g8f8 e8b5 f7f5 e5f6 g7f6 b6c5 b4c5 c1c5 h6h5 c5c7 e4g6 h2h4 g6f7 f2f4 f8g7 b5e8 e6e5 e8f7 d5d4 f7h5 g7h8 e3d4 e5e4 g1f1 e4e3 f1e2 f6f5 e2e3 h8g8 e3d3 g8h8 d3c4 h8g8 c4b5 g8h8 b5b6 h8g8 b6b7 g8h8 b7a8 h8g8 a8a7 g8h8 a7a6 h8g8 a6a5 g8h8 a5a4 h8g8 a4a3 g8h8 a3a2 h8g8 a2a1 g8h8 a1b1 h8g8 b1c1 g8h8 c1d1 h8g8 d1e1 g8h8 e1f1 h8g8 f1g1 g8h8 g1h2 h8g8 h2g3 g8h8 g3f3 h8g8 f3e3 g8h8 e3d3 h8g8 d3c3 g8h8 c3b3 h8g8 b3b4 g8h8 b4b5 h8g8 h5g6 g8h8 g6f5 h8g8 b5b6 g8h8 b6b7 h8g8 b7a7 g8h8 a7a6 h8g8 a6a5 g8h8 a5a4 h8g8 a4a3 g8h8 a3a2 h8g8 a2a1 g8h8 a1b1 h8g8 b1c1 g8h8 c1d1 h8g8 d1e1 g8h8 e1f1 h8g8 f1g1 g8h8 g1h1 h8g8 h1h2 g8h8 h2h3 h8g8 h3g3 g8h8 g3f2 h8g8 f2e2 g8h8 e2d2 h8g8 d2c2 g8h8 c2b2 h8g8 b2b3 g8h8 b3b4 

In [8]:
# count number of <GAME> and </GAME> tokens in pretrain parquet
flat_tokens = pl.col("token_ids").explode()
counts = pretrain_sample.select(
    flat_tokens.eq(GAME_START_ID).sum().alias("game_start_count"),
    flat_tokens.eq(GAME_END_ID).sum().alias("game_end_count"),
)

print("Tokenized column checked: token_ids")
print("<GAME> token ID:", GAME_START_ID)
print("</GAME> token ID:", GAME_END_ID)
print("total <GAME> tokens in sample:", int(counts["game_start_count"][0]))
print("total </GAME> tokens in sample:", int(counts["game_end_count"][0]))

Tokenized column checked: token_ids
<GAME> token ID: 0
</GAME> token ID: 1
total <GAME> tokens in sample: 1000
total </GAME> tokens in sample: 1000


### Tokenized game example


In [11]:
example_game = pretrain_sample.head(1).to_dicts()[0]
# or Pick shortest game from pretrain_sample
# example_game = pretrain_sample.sort(pl.col("token_ids").list.len()).head(1).to_dicts()[0]
token_ids = example_game["token_ids"]

print("Token IDs:", token_ids)
print("Total tokens:", len(token_ids))

Token IDs: [0, 3, 12, 12, 1235, 3636, 859, 1880, 3563, 2312, 823, 614, 1201, 1502, 1625, 1406, 1959, 3140, 1591, 20, 26, 27, 83, 99, 1906, 3069, 1118, 1253, 1020, 3483, 3340, 20, 22, 973, 27, 36, 28, 1150, 3169, 410, 3389, 1540, 877, 2286, 27, 54, 28, 143, 1836, 3295, 1498, 1251, 1788, 1281, 1070, 20, 23, 1615, 1176, 3091, 27, 83, 99, 2952, 657, 1558, 547, 16, 18, 2228, 599, 1254, 3145, 27, 64, 28, 522, 3275, 164, 27, 88, 28, 1699, 16, 17, 2654, 3103, 16, 18, 982, 1917, 852, 27, 90, 28, 1985, 378, 2271, 3126, 145, 742, 559, 1054, 1299, 3316, 481, 2560, 20, 25, 119, 2204, 20, 25, 547, 3166, 619, 2936, 313, 3316, 463, 652, 867, 16, 17, 2654, 27, 90, 28, 547, 27, 60, 28, 1]
Total tokens: 133


In [12]:
# Same game decoded as string tokens
tokens_decoded = [ID_TO_MOVE.get(tid, f"<{tid}>") for tid in token_ids]

print("Tokens as strings:")
for i, token in enumerate(tokens_decoded):
    print(f"  {i}: {token}")

print(f"\nTotal tokens: {len(tokens_decoded)}")

Tokens as strings:
  0: <game_start>
  1: <white_won>
  2: <elo_2000_2499>
  3: <elo_2000_2499>
  4: w:d2d4
  5: b:g8f6
  6: w:c2c4
  7: b:e7e6
  8: w:g1f3
  9: b:f8b4
  10: w:c1d2
  11: b:b4d2
  12: w:d1d2
  13: b:d7d5
  14: w:e2e3
  15: b:d5c4
  16: w:f1c4
  17: b:b8d7
  18: w:e1g1
  19: <what_piece>
  20: <king>
  21: <what_is_on>
  22: <g7>
  23: <b:pawn>
  24: b:e8g8
  25: w:b1c3
  26: b:c7c5
  27: w:d2c2
  28: b:c5d4
  29: w:f3d4
  30: b:d7b6
  31: <what_piece>
  32: <knight>
  33: w:c4d3
  34: <what_is_on>
  35: <h1>
  36: <empty>
  37: b:c8d7
  38: w:c3e4
  39: b:a8c8
  40: w:e4f6
  41: b:d8f6
  42: w:c2d2
  43: b:f8d8
  44: <what_is_on>
  45: <b4>
  46: <empty>
  47: w:a1c1
  48: b:e6e5
  49: w:d4b3
  50: b:d7c6
  51: w:d2e2
  52: b:e5e4
  53: w:d3b5
  54: b:c6b5
  55: <what_piece>
  56: <bishop>
  57: w:e2b5
  58: b:c8c1
  59: w:b3c1
  60: <what_is_on>
  61: <g7>
  62: <b:pawn>
  63: b:h7h6
  64: w:b5b3
  65: b:d8d2
  66: w:b3b4
  67: <is_check>
  68: <no_check>
  69: b:f6b2
